In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import numpy as np
import datetime
import time
import pickle
import os
import seaborn as sns
from scipy.interpolate import make_interp_spline
from tabulate import tabulate
import sys
sys.path.append("src")
from data_generation import *
from neural_networks import *
from estimation import *

pd.options.mode.chained_assignment = None
os.chdir('Result_Tables/')

In [2]:
def feature_generation_endo(J, K, M, seed):
    # J is number of products per markets (assume each market has the same number of products but different products)
    # K is number of features (excluding price)
    # M is the number of markets (assume each market has different sets of products)
    np.random.seed(seed)

    Total_J = J * M
    x_1 = np.random.normal(loc=0, scale=1, size=(Total_J , K))
    mu = np.random.normal(loc=0, scale=1, size = Total_J)
    z = np.random.uniform(0,4,size = Total_J)
    price = z + mu
    
    X = pd.DataFrame(x_1)
    X['price'] = price

    return X, mu, z 

In [10]:
def endo_iteration(mode, J, M, K, seed, dg, params, datax, delta):
    setup = [J, K, M, str(dg).split()[1], seed]
    data_train, data_test = split_train_test(datax, p = 0.8)
    ### step 2: train models 
    m1_deep, loss_deep = train_deep(data_train)
     
    ### step 3: evaluation
    data = data_train.copy()

    y_pred_deep1 = pred_deep(data, m1_deep)
    error_deep1 = get_errors(y_pred_deep1, data['Y'])
    
    train_pred_df = pd.DataFrame({'true':data['Y'], 'pred_deep':y_pred_deep1})
    
    train_pred_df.to_csv(mode + "_train"+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".csv")

    print("J=", data['J'],"K=", data['K'],"M=", data['M'], "\n", str(dg).split()[1], 'training error',
          "deep error: ", error_deep1[1], "\n") 
    
    data = data_test.copy()

    y_pred_deep1 = pred_deep(data, m1_deep)
    error_deep1 = get_errors(y_pred_deep1, data['Y'])
    
    
    print("J=", data['J'],"K=", data['K'],"M=", data['M'], "\n",  str(dg).split()[1], 'test error'
          "deep error: ", error_deep1[1], "\n") 
    
    test_pred_df = pd.DataFrame({'true':data['Y'], 'pred_deep':y_pred_deep1})
    
    test_pred_df.to_csv(mode + "_test"+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".csv")
    
    error_all = setup + error_deep1 
    error_file_name = mode + '_error_'+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".pickle"
    
    with open(error_file_name, "wb") as f:
        # dump the dictionary into the file using pickle.dump()
        pickle.dump(error_all, f)
    
    
    model_list = [m1_deep]
    model_file_name = mode + '_model_'+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".pickle"
    
    with open(model_file_name, "wb") as f:
        # dump the dictionary into the file using pickle.dump()
        pickle.dump(model_list, f)
        

    ### step 5: elasticity
    J = data_train['J']
    large_record = pd.DataFrame(columns = ['market_id','i','j','old_price','new_price','type',
                                           'true_elasticity','pred_elasticity_deep','price_i'])
    for prod_id in range(J):
        data_train = cal_true_elasticity(data_train, dg, prod_id, delta, seed)
        record = cal_elasticity_record(data_train, pred_deep, model_list[0], prod_id, delta)
        record = record.rename(columns={'pred_elasticity': 'pred_elasticity_deep'})
        record['i'] = prod_id
        record['j'] = record.index % data_train['J']
        large_record = pd.concat([large_record, record]).reset_index(drop=True)

    large_record['error_deep'] = large_record['pred_elasticity_deep'] - large_record['true_elasticity']
    
    large_record.to_csv(mode + '_elas_record_'+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".csv", index = False)
    
    elas_all = large_record.copy()
    elas_all['ae_deep'] = np.abs(elas_all['error_deep'])
    
    elas_all.to_csv(mode + '_elas_all' + 'J' + str(J) + 'K' +  str(K) +'M' + str(M)+str(dg).split()[1] +'.csv', index = False)
    
    elas_valid = elas_all.loc[(elas_all.true_elasticity.isna()==False) & (np.isfinite(elas_all.true_elasticity) == True)].copy()
    
    elas_rslt_median = elas_valid.groupby(['type'])[['ae_deep']].median().reset_index()
    elas_rslt_mean = elas_valid.groupby(['type'])[['ae_deep']].mean().reset_index()
    
    print('elas_mean', elas_rslt_mean)
    print('elas_median', elas_rslt_median)
    
    return 1 

In [8]:
def endo_data_generation(J, K, M, seed, dg, params):
    X, mu, z = feature_generation_endo(J, K, M, seed)
            
    X_comp = X.copy()
    X_comp.insert(loc=K, column=K, value=mu)

    iv_model = sm.OLS(X['price'], z).fit()
    residuals = iv_model.resid

    X_endo = X.copy()
    X_endo.insert(loc=K, column=K, value= residuals ) 

    data_comp = dg(X_comp, params, J, K+1, M, seed)
    data_endo = data_comp.copy()
    data_endo['X'] = X_endo
    
    data_wrong = data_comp.copy()
    data_wrong['X'] = X.copy()
    data_wrong['K'] = K
    
    return data_endo, data_comp, data_wrong

In [28]:
def endo_report(J, K, M, seed_list, dg):
    
    elas_all = pd.DataFrame()
    error_df = pd.DataFrame(columns = ['J', 'K', 'M', 'dg', 'seed',"mse_deep","mae_deep"])
    o_K = K
    for mode in ['endo','comp','wrong']:
        for seed in seed_list:
            if mode !='wrong':
                K = o_K+1
            else:
                K = o_K
            
            error_file = mode + '_error_'+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".pickle"
            elas_file = mode + '_elas_record_'+ 'J' + str(J) + 'K' +  str(K) +'M' + str(M) + '_'+str(seed) + str(dg).split()[1] +".csv"
            elas_record = pd.read_csv(elas_file)
            elas_record['seed'] = seed
            with open(error_file, 'rb') as f:
                error_all = pickle.load(f) 

            error_df = pd.concat([error_df, pd.DataFrame([error_all], columns= error_df.columns)])
            elas_all = pd.concat([elas_all, elas_record])
            

        error_df.to_csv(mode + '_error_df' + 'J' + str(J) + 'K' +  str(K) +'M' + str(M)+str(dg).split()[1] +'.csv')
        elas_all['ae_deep'] = np.abs(elas_all['error_deep'])

        elas_all.to_csv(mode + '_elas_all' + 'J' + str(J) + 'K' +  str(K) +'M' + str(M)+str(dg).split()[1] +'.csv', index = False)
    
        elas_valid = elas_all.loc[(elas_all.true_elasticity.isna()==False) & (np.isfinite(elas_all.true_elasticity) == True)].copy()

        elas_rslt_mean = elas_valid.groupby(['type'])[['ae_deep']].mean().reset_index()

        elas_rslt_mean.to_csv(mode+'_elas_mae_' + 'J' + str(J) + 'K' +  str(K) +'M' + str(M)+str(dg).split()[1] +'.csv', index = False)

    return 1

In [5]:
seed_list = [7456,145609, 214341, 237639, 148649, 763490, 677253, 560404, 591830,
    84947,  26672, 184535, 874369,  39876, 738836, 100589, 812811,
    75122, 182935, 911677]

dg_list = [rcl] 
JKM_list = [[10,10,100]]
delta = 0.1 

hyper_params = {'hidden_size': 256, 'num_hidden_layers': 5, 'n_epochs': 2000, 'learning_rate': 0.0001}

os.chdir("/mnt/sdb1/colab-project/rslt_0414/endo")

In [11]:
for [J, K, M] in JKM_list:
    for dg in dg_list:
        for i in range(len(seed_list)):
            seed = seed_list[i]
            params = [np.random.normal(0,1,K+3) / (K * 2), np.ones(K+3)]
            params[0][-1] = -1
            data_endo, data_comp, data_wrong = endo_data_generation(J, K, M, seed, dg, params)
            endo_iteration('endo',J, M, K+1, seed, dg, params, data_endo, delta)
            endo_iteration('comp',J, M, K+1, seed, dg, params, data_comp, delta)
            endo_iteration('wrong',J, M, K, seed, dg, params, data_wrong, delta)
            print(J, K, M, i, 'finished', datetime.datetime.now())

J= 10 K= 11 M= 80 
 rcl training error deep error:  9.981734352400576e-05 

J= 10 K= 11 M= 20 
 rcl test errordeep error:  0.0265406391651378 

elas_mean     type  ae_deep
0  cross   0.0591
1   self   0.2159
elas_median     type  ae_deep
0  cross   0.0376
1   self   0.1304
J= 10 K= 11 M= 80 
 rcl training error deep error:  7.231927437927537e-05 

J= 10 K= 11 M= 20 
 rcl test errordeep error:  0.031221956528521066 

elas_mean     type  ae_deep
0  cross   0.0665
1   self   0.2302
elas_median     type  ae_deep
0  cross   0.0400
1   self   0.1491
J= 10 K= 10 M= 80 
 rcl training error deep error:  0.00018633844655526396 

J= 10 K= 10 M= 20 
 rcl test errordeep error:  0.030513544192329503 

elas_mean     type  ae_deep
0  cross   5.7926
1   self   6.6685
elas_median     type  ae_deep
0  cross   4.0335
1   self   4.3525
10 10 100 0 finished 2024-02-11 18:44:21.848654
J= 10 K= 11 M= 80 
 rcl training error deep error:  8.116366497350936e-05 

J= 10 K= 11 M= 20 
 rcl test errordeep error:  0.

FileNotFoundError: [Errno 2] No such file or directory: 'elas_record_J10K10M100_7456rcl.csv'

In [29]:
for [J, K, M] in JKM_list:
    for dg in dg_list:
        endo_report(J, K, M, seed_list, dg)

In [50]:
mode = 'comp'
o_K = 10
for mode in ['endo','comp','wrong']:
    for [J, K, M] in JKM_list:
        for dg in dg_list:
            if mode !='wrong':
                K = o_K+1
            else:
                K = o_K
            elas_all = pd.read_csv(mode + '_elas_all' + 'J' + str(J) + 'K' +  str(K) +'M' + str(M)+str(dg).split()[1] +'.csv')
            elas_valid = elas_all.loc[(elas_all.true_elasticity.isna()==False) & (np.isfinite(elas_all.true_elasticity) == True)].copy()
            elas_rslt_std = elas_valid.groupby(['type'])[['error_deep']].std().reset_index()
            print(mode, elas_rslt_std)

endo     type  error_deep
0  cross      0.1033
1   self      0.3516
comp     type  error_deep
0  cross      0.1039
1   self      0.3469
wrong     type  error_deep
0  cross      4.4306
1   self      5.1901


In [49]:
mode = 'comp'
o_K = 10
for mode in ['endo','comp','wrong']:
    for [J, K, M] in JKM_list:
        for dg in dg_list:
            if mode !='wrong':
                K = o_K+1
            else:
                K = o_K
            path = mode+'_elas_mae_' + 'J' + str(J) + 'K' +  str(K) +'M' + str(M)+str(dg).split()[1] +'.csv'
            df = pd.read_csv(path)
            print(mode, df)

endo     type  ae_deep
0  cross   0.0669
1   self   0.2307
comp     type  ae_deep
0  cross   0.0657
1   self   0.2290
wrong     type  ae_deep
0  cross   1.9751
1   self   2.3832


In [48]:
for mode in ['endo','comp','wrong']:
      
    if mode !='wrong':
        K = o_K+1
    else:
        K = o_K
            
    path = mode + '_error_df' + 'J' + str(J) + 'K' +  str(K) +'M' + str(M)+str(dg).split()[1] +'.csv'
    df = pd.read_csv(path)
    print(mode, df.mean())

endo Unnamed: 0         0.0000
J                 10.0000
K                 11.0000
M                100.0000
seed          368952.0000
mse_deep           0.0194
mae_deep           0.0256
dtype: float64
comp Unnamed: 0         0.0000
J                 10.0000
K                 11.0000
M                100.0000
seed          368952.0000
mse_deep           0.0195
mae_deep           0.0254
dtype: float64
wrong Unnamed: 0         0.0000
J                 10.0000
K                 10.6667
M                100.0000
seed          368952.0000
mse_deep           0.0202
mae_deep           0.0263
dtype: float64


/tmp/ipykernel_444328/4197927551.py:10: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  print(mode, df.mean())
/tmp/ipykernel_444328/4197927551.py:10: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  print(mode, df.mean())
/tmp/ipykernel_444328/4197927551.py:10: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  

In [41]:
df #comp

,type,ae_deep
0,cross,0.0657
1,self,0.2290


In [39]:
df#wrong

,type,ae_deep
0,cross,1.9751
1,self,2.3832


In [37]:
df#endo

,type,ae_deep
0,cross,0.0669
1,self,0.2307
